first attempt using gridsearch and not in depth parameters for lightgbm and catboost url : https://github.com/tyleroberts4/Advanced-Machine-Learning/blob/main/Kaggle/Homework4.ipynb

This model ended up being my best, I used a gridsearch but of the tuning paramters that were the best from optuna tuning last week, also lightgbm and catboost since they were best performing on average. url: https://github.com/tyleroberts4/Advanced-Machine-Learning/blob/main/Kaggle/Homework4_extra.ipynb

## Approach Discussion

I tried two main versions of the Homework 4 workflow. The first notebook, `Homework4.ipynb`, used two tuned models: CatBoost and LightGBM. CatBoost used the raw categorical variables directly, while LightGBM used an engineered representation with polynomial numeric features and target-encoded categorical features. I then compared the individual models to a simple probability average and a stacked model using the two models' predicted probabilities as meta-features.

In `Homework4_extra.ipynb`, I kept the same two-model structure but added extra feature engineering inspired by Homework 3. These included log transformations, squared temperature, humidity-temperature interactions, quantile bins, and binary high/low indicators. I also changed the GridSearch parameter grids so they were centered around the best Optuna parameter settings from my Homework 3 CatBoost and LightGBM tuning notebooks.

| Submission / Approach | Leaderboard Balanced Accuracy |
|---|---:|
| CatBoost with Feature Engineering | 0.95920 |
| LightGBM and CatBoost Stacked - best run | 0.96232 |
| LightGBM and CatBoost stacked not many features and parameters` - earlier run | 0.96223 |
| CatBoost Optuna tuning | 0.95888 |
| LightGB< optuna tuning> | 0.95958 |


Overall, the strongest idea was using two different model representations. CatBoost worked well because it handled categorical variables naturally, while LightGBM benefited from numeric feature engineering and target encoding. The stacking approach was useful because the two models made predictions from different feature views, so the meta-model had a chance to combine complementary information.

### Feature Importance and How It Affected Feature Use

Permutation importance helped confirm that the strongest signal was coming from a small group of weather-related variables, not from every engineered feature. The most important features were very consistent across LightGBM and CatBoost.

| Model | Top Permutation-Important Features | Approx. Permutation Importance |
|---|---|---:|
| LightGBM | `Soil_Moisture` | 0.33357 |
| LightGBM | `Crop_Growth_Stage__target_High_te` | 0.28201 |
| LightGBM | `Temperature_C` | 0.18524 |
| LightGBM | `Mulching_Used__target_High_te` | 0.16552 |
| LightGBM | `Soil_Moisture Rainfall_mm` | 0.00810 |
| LightGBM | `Temperature_C Rainfall_mm` | 0.00470 |
| CatBoost | `Soil_Moisture` | 0.34288 |
| CatBoost | `Crop_Growth_Stage` | 0.29597 |
| CatBoost | `Temperature_C` | 0.19872 |
| CatBoost | `Mulching_Used` | 0.18505 |

The permutation results showed that `Soil_Moisture`, `Crop_Growth_Stage`, `Temperature_C`, and `Mulching_Used` were the most useful predictors. This made sense for the problem because irrigation need should depend heavily on soil water content, crop development stage, temperature stress, and whether mulching helps retain moisture.

For LightGBM, the target-encoded categorical features were especially useful. For example, `Crop_Growth_Stage__target_High_te` and `Mulching_Used__target_High_te` ranked near the top, which supports using leakage-safe target encoding for categorical variables. Some polynomial interaction terms also appeared useful, such as `Soil_Moisture Rainfall_mm` and `Temperature_C Rainfall_mm`, but their importance was much smaller than the main effects. This suggests that interactions helped a little, but most of the predictive power came from the original high-signal variables.

This affected my feature engineering approach. I kept the core domain features and target encodings because they aligned with the permutation results. I also kept some interaction features because they added additional signal for LightGBM, but I would not keep adding many more polynomial terms blindly. Moving forward, I would focus feature engineering around moisture, crop stage, temperature, rainfall, and mulching, rather than creating a large number of unrelated features.

Overall, the model improvements from feature engineering and stacking were meaningful but not huge. The leaderboard score of **0.96232 balanced accuracy** with rank **2231** suggests the final approach was competitive, but the gains are likely incremental because CatBoost and LightGBM were already strong models. The permutation importance results support this: a few important variables drove most of the performance, while extra engineered features mostly provided smaller refinements.

The extra engineered features were worth trying because they tested different representations of the data, especially interactions and grouped/bin features. However, I would not assume all of them help. Some may add noise or duplicate information already captured by tree models. The feature importance and permutation importance sections are useful for deciding which engineered features actually matter.
